| 컬럼명 | 설명 |
|---|---|
| index | 행 인덱스 |
| course_id | 강의 ID |
| userid_DI | 학습자 식별자 |
| registered | 등록 여부 |
| viewed | 강의를 열어본 여부 |
| explored | 강의를 적극적으로 탐색했는지 여부 |
| certified | 인증서 획득 여부 |
| final_cc_cname_DI | 국가 |
| LoE_DI | 학력 수준 (Level of Education) |
| YoB | 출생연도 |
| gender | 성별 |
| grade | 성적 |
| start_time_DI | 강의 시작일 |
| last_event_DI | 마지막 활동일 |
| nevents | 전체 이벤트 수 |
| ndays_act | 실제 활동 일수 |
| nplay_video | 영상 재생 수 |
| nchapters | 탐색한 챕터 수 |
| nforum_posts | 포럼 게시글 수 |
| roles | 사용자 역할 |
| incomplete_flag | 불완전 데이터 관련 플래그 |

In [20]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


라이브러리 로드 완료!
한글 폰트 설정 완료!


In [21]:
# 데이터 로드
df = pd.read_csv('../../data/Courses.csv')

In [22]:
print(df.shape)
df.info()

(641138, 21)
<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   index              641138 non-null  int64  
 1   course_id          641138 non-null  str    
 2   userid_DI          641138 non-null  str    
 3   registered         641138 non-null  int64  
 4   viewed             641138 non-null  int64  
 5   explored           641138 non-null  int64  
 6   certified          641138 non-null  int64  
 7   final_cc_cname_DI  641138 non-null  str    
 8   LoE_DI             535130 non-null  str    
 9   YoB                544533 non-null  float64
 10  gender             554332 non-null  str    
 11  grade              592766 non-null  str    
 12  start_time_DI      641138 non-null  str    
 13  last_event_DI      462184 non-null  str    
 14  nevents            441987 non-null  float64
 15  ndays_act          478395 non-null  float64
 16  

In [23]:
# 불필요한 컬럼 제거
df.drop(columns='roles', inplace=True)
print(df.shape)
df.info()

(641138, 20)
<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 20 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   index              641138 non-null  int64  
 1   course_id          641138 non-null  str    
 2   userid_DI          641138 non-null  str    
 3   registered         641138 non-null  int64  
 4   viewed             641138 non-null  int64  
 5   explored           641138 non-null  int64  
 6   certified          641138 non-null  int64  
 7   final_cc_cname_DI  641138 non-null  str    
 8   LoE_DI             535130 non-null  str    
 9   YoB                544533 non-null  float64
 10  gender             554332 non-null  str    
 11  grade              592766 non-null  str    
 12  start_time_DI      641138 non-null  str    
 13  last_event_DI      462184 non-null  str    
 14  nevents            441987 non-null  float64
 15  ndays_act          478395 non-null  float64
 16  

In [24]:
# 컬럼 전체 소문자 변경
df.columns = df.columns.str.lower()

# 컬럼명 변경
df.rename(columns={'userid_di': 'userid', 'final_cc_cname_di': 'country', 'loe_di': 'loe', 'start_time_di': 'start_time', 'last_event_di': 'last_event'}, inplace=True)

# 결과확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 20 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            641138 non-null  int64  
 1   course_id        641138 non-null  str    
 2   userid           641138 non-null  str    
 3   registered       641138 non-null  int64  
 4   viewed           641138 non-null  int64  
 5   explored         641138 non-null  int64  
 6   certified        641138 non-null  int64  
 7   country          641138 non-null  str    
 8   loe              535130 non-null  str    
 9   yob              544533 non-null  float64
 10  gender           554332 non-null  str    
 11  grade            592766 non-null  str    
 12  start_time       641138 non-null  str    
 13  last_event       462184 non-null  str    
 14  nevents          441987 non-null  float64
 15  ndays_act        478395 non-null  float64
 16  nplay_video      183608 non-null  float64
 17  nc

In [25]:
# str => date 타입 변경
df['start_time'] = pd.to_datetime(df['start_time'])
df['last_event'] = pd.to_datetime(df['last_event'])

# str => float 타입 변경
df['grade'] = pd.to_numeric(df['grade'], errors='coerce')

# float => int 타입 변환
df[['nevents', 'ndays_act', 'nplay_video', 'nchapters', 'incomplete_flag', 'yob']] = df[['nevents', 'ndays_act', 'nplay_video', 'nchapters', 'incomplete_flag', 'yob']].astype('Int64')

# data 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 641138 entries, 0 to 641137
Data columns (total 20 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   index            641138 non-null  int64         
 1   course_id        641138 non-null  str           
 2   userid           641138 non-null  str           
 3   registered       641138 non-null  int64         
 4   viewed           641138 non-null  int64         
 5   explored         641138 non-null  int64         
 6   certified        641138 non-null  int64         
 7   country          641138 non-null  str           
 8   loe              535130 non-null  str           
 9   yob              544533 non-null  Int64         
 10  gender           554332 non-null  str           
 11  grade            583738 non-null  float64       
 12  start_time       641138 non-null  datetime64[us]
 13  last_event       462184 non-null  datetime64[us]
 14  nevents          441987 non-nul

In [26]:
# 데이터 소문자 변경
df['country'] = df['country'].str.lower()
df['userid'] = df['userid'].str.lower()
df['loe'] = df['loe'].str.lower()
df['course_id'] = df['course_id'].str.lower()
df['gender'] = df['gender'].str.lower()
df.head(5)

,index,course_id,userid,registered,viewed,explored,certified,country,loe,yob,gender,grade,start_time,last_event,nevents,ndays_act,nplay_video,nchapters,nforum_posts,incomplete_flag
0,0,harvardx/cb22x/2013_spring,mhxpc130442623,1,0,0,0,united states,NaN,<NA>,NaN,0.0,2012-12-19,2013-11-17,<NA>,9,<NA>,<NA>,0,1
1,1,harvardx/cs50x/2012,mhxpc130442623,1,1,0,0,united states,NaN,<NA>,NaN,0.0,2012-10-15,NaT,<NA>,9,<NA>,1,0,1
2,2,harvardx/cb22x/2013_spring,mhxpc130275857,1,0,0,0,united states,NaN,<NA>,NaN,0.0,2013-02-08,2013-11-17,<NA>,16,<NA>,<NA>,0,1
3,3,harvardx/cs50x/2012,mhxpc130275857,1,0,0,0,united states,NaN,<NA>,NaN,0.0,2012-09-17,NaT,<NA>,16,<NA>,<NA>,0,1
4,4,harvardx/er22x/2013_spring,mhxpc130275857,1,0,0,0,united states,NaN,<NA>,NaN,0.0,2012-12-19,NaT,<NA>,16,<NA>,<NA>,0,1


In [27]:
print("\n" + "="*60)
print("기초 통계")
print("="*60)
display(df.describe(include='all').T)


기초 통계


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
index,641138.0,NaN,NaN,NaN,320568.5,0.0,160284.25,320568.5,480852.75,641137.0,185080.742781
course_id,641138,16,harvardx/cs50x/2012,169621,NaN,NaN,NaN,NaN,NaN,NaN,NaN
userid,641138,476532,mhxpc130505428,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registered,641138.0,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,0.0
viewed,641138.0,NaN,NaN,NaN,0.624299,0.0,0.0,1.0,1.0,1.0,0.484304
explored,641138.0,NaN,NaN,NaN,0.061899,0.0,0.0,0.0,0.0,1.0,0.240973
certified,641138.0,NaN,NaN,NaN,0.027587,0.0,0.0,0.0,0.0,1.0,0.163786
country,641138,34,united states,184240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loe,535130,5,bachelor's,219768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
yob,544533.0,<NA>,<NA>,<NA>,1985.253279,1931.0,1982.0,1988.0,1991.0,2013.0,8.891814


In [28]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측비율(%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")



결측치 확인

[결측치 현황]


,결측수,결측비율(%)
incomplete_flag,540977,84.38
nplay_video,457530,71.36
nchapters,258753,40.36
nevents,199151,31.06
last_event,178954,27.91
ndays_act,162743,25.38
loe,106008,16.53
yob,96605,15.07
gender,86806,13.54
grade,57400,8.95


In [29]:
# 결측치 대체
df['grade'] = df['grade'].fillna(0)
df['loe'] = df['loe'].fillna('unknown')
df['gender'] = df['gender'].fillna('unknown')

In [30]:
# 이상치 대체
# o => unknown
# 개수 확인
print("="*60)
print("gender 값이 o => unknown 대체")
print("="*60)

target_count = df.loc[df['gender'] == 'o', 'gender'].count()
print(f"대체 전 대상 개수: {target_count} 개")

df.loc[df['gender'] == 'o', 'gender'] = 'unknown'
replaced_count = df.loc[df['gender'] == 'unknown', 'gender'].count()

print(f"대체 완료!")
print(f"'unknown' 개수: {replaced_count} 개")

gender 값이 o => unknown 대체
대체 전 대상 개수: 17 개
대체 완료!
'unknown' 개수: 86823 개


In [31]:
# grade = 0 & certified = 1 인 값 확인 및 제거
# mhxpc130066792
print("="*60)
print("grade = 0 & certified = 1 인 값 확인 및 제거")
print("="*60)

target_user_count = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid'].count()
print(f"삭제 전: {target_user_count} 개")

target_users = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# 제거 여부 확인
exists_count = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid'].count()
print(f"삭제 후: {exists_count} 개")

# 데이터 수 확인
print(df.shape)

grade = 0 & certified = 1 인 값 확인 및 제거
삭제 전: 1 개
삭제 후: 0 개
(641137, 20)


In [32]:
print("="*60)
print("viewed = 0 & explored = 1 인 값 확인 및 제거")
print("="*60)

target_user_count = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid'].count()
print(f"삭제 전: {target_user_count} 개")

target_users = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# 제거 여부 확인
exists_count = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid'].count()
print(f"삭제 후: {exists_count} 개")

# 데이터 수 확인
print(df.shape)

viewed = 0 & explored = 1 인 값 확인 및 제거
삭제 전: 8 개
삭제 후: 0 개
(641112, 20)


In [33]:
print("="*60)
print("last_event 결측값 대체")
print("="*60)

na_cnt = df['last_event'].isna().sum()
print(f"결측 수: {na_cnt}개")

# duration 파생컬럼 생성
df['duration'] = (df['last_event'] - df['start_time']).dt.days

# start_time 이 last_event 보다 큰 이상치 조회 => last_event - start_time 중앙값 계산에서 제외
excepted_users = df.loc[(df['start_time'] > df['last_event']), 'userid']

# 강의 + 활동 별 (last_event - start_event) 중앙값 대체 (이상치 제외)
group_cols = ['course_id', 'registered', 'viewed', 'explored', 'certified']
df_filtered = df[~df['userid'].isin(excepted_users)]
group_duration = df_filtered.groupby(group_cols)['duration'].median().reset_index()
group_duration.rename(columns={'duration': 'duration_median'}, inplace=True)
display(group_duration)

# last_event 결측치를 그룹 중앙값으로 대체
df = df.merge(group_duration, on=group_cols, how='left')
mask = df['last_event'].isna()
df.loc[mask, 'last_event'] = (
    pd.to_datetime(df.loc[mask, 'start_time']) +
    pd.to_timedelta(df.loc[mask, 'duration_median'], unit='D')
).dt.date
df.drop(columns='duration_median', inplace=True)

print(f"\n대체 후 결측 수: {df['last_event'].isna().sum()}개")
print(df.shape)

last_event 결측값 대체
결측 수: 178949개


,course_id,registered,viewed,explored,certified,duration_median
0,harvardx/cb22x/2013_spring,1,0,0,0,142.0
1,harvardx/cb22x/2013_spring,1,1,0,0,25.0
2,harvardx/cb22x/2013_spring,1,1,0,1,168.5
3,harvardx/cb22x/2013_spring,1,1,1,0,117.0
4,harvardx/cb22x/2013_spring,1,1,1,1,160.0
...,...,...,...,...,...,...
73,mitx/8.mrev/2013_summer,1,0,0,0,0.0
74,mitx/8.mrev/2013_summer,1,1,0,0,15.0
75,mitx/8.mrev/2013_summer,1,1,0,1,77.0
76,mitx/8.mrev/2013_summer,1,1,1,0,83.0



대체 후 결측 수: 0개
(641112, 21)


In [34]:
# 13세 미만 사용자 제거
print("="*40)
print("13세 미만 사용자 제거")
print("="*40)
under_12 = df.loc[(pd.to_datetime(df['start_time']).dt.year - df['yob'] < 13), 'userid'].count()
print(f"13세 미만 사용자 수: {under_12}")
df['age'] = pd.to_datetime(df['start_time']).dt.year - df['yob']
df = df[df['age'] >= 13]
print(f"제거완료!")
print(df.shape)

13세 미만 사용자 제거
13세 미만 사용자 수: 945
제거완료!
(543580, 22)


In [35]:
# 파생컬럼 생성
# course_id 소문자 변경 후 기관/강의코드/강의년도/강의학기 생성
df[['institution', 'course_code', 'year_term']] = df['course_id'].str.split('/', expand=True)
df[['course_year', 'course_term']] = df['year_term'].str.split('_', expand=True)
df['course_term'] = df['course_term'].fillna('unknown')
df.drop(columns='year_term', inplace=True)

# course_id 소문자 기준으로 카테고리 매핑
category_map = {
    'harvardx/cb22x/2013_spring': '문학',
    'harvardx/cs50x/2012': '컴퓨터프로그래밍',
    'harvardx/er22x/2013_spring': '인문학',
    'harvardx/ph207x/2012_fall': '의료서비스',
    'harvardx/ph278x/2013_spring': '환경과학',
    'mitx/14.73x/2013_spring': '경제학',
    'mitx/2.01x/2013_spring': '공학',
    'mitx/3.091x/2012_fall': '화학',
    'mitx/3.091x/2013_spring': '화학',
    'mitx/6.002x/2012_fall': '공학',
    'mitx/6.002x/2013_spring': '공학',
    'mitx/6.00x/2012_fall': '컴퓨터프로그래밍',
    'mitx/6.00x/2013_spring': '컴퓨터프로그래밍',
    'mitx/7.00x/2013_spring': '생물학',
    'mitx/8.02x/2013_spring': '물리학',
    'mitx/8.mrev/2013_summer': '물리학',
}

df['course_category'] = df['course_id'].map(category_map)
df.info()

<class 'pandas.DataFrame'>
Index: 543580 entries, 19330 to 641111
Data columns (total 27 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   index            543580 non-null  int64         
 1   course_id        543580 non-null  str           
 2   userid           543580 non-null  str           
 3   registered       543580 non-null  int64         
 4   viewed           543580 non-null  int64         
 5   explored         543580 non-null  int64         
 6   certified        543580 non-null  int64         
 7   country          543580 non-null  str           
 8   loe              543580 non-null  str           
 9   yob              543580 non-null  Int64         
 10  gender           543580 non-null  str           
 11  grade            543580 non-null  float64       
 12  start_time       543580 non-null  datetime64[us]
 13  last_event       543580 non-null  datetime64[us]
 14  nevents          370031 non-null

In [36]:
# 전처리 완료 데이터 생성
df.to_csv('../../data/preprocessed.csv', index=False)

In [37]:
processed_data = pd.read_csv('../../data/preprocessed.csv')
print(processed_data.shape)
processed_data.info()

(543580, 27)
<class 'pandas.DataFrame'>
RangeIndex: 543580 entries, 0 to 543579
Data columns (total 27 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   index            543580 non-null  int64  
 1   course_id        543580 non-null  str    
 2   userid           543580 non-null  str    
 3   registered       543580 non-null  int64  
 4   viewed           543580 non-null  int64  
 5   explored         543580 non-null  int64  
 6   certified        543580 non-null  int64  
 7   country          543580 non-null  str    
 8   loe              543580 non-null  str    
 9   yob              543580 non-null  int64  
 10  gender           543580 non-null  str    
 11  grade            543580 non-null  float64
 12  start_time       543580 non-null  str    
 13  last_event       543580 non-null  str    
 14  nevents          370031 non-null  float64
 15  ndays_act        398876 non-null  float64
 16  nplay_video      149337 non-null  fl